#  Agentic YouTube Video Quality Analysis
## Notebook 01 — Collecte de Données (YouTube Data API v3)
## Contexte : Vidéos Éducatives Cameroun (3ème, Première, Terminale)

**Objectif** : Collecter des vidéos YouTube et leurs commentaires sur les matières scolaires
camerounaises (systèmes francophone et anglophone), avec optimisation stricte du quota API.

**Systèmes scolaires** : Francophone (BEPC/Bac) + Anglophone (GCE O/A-Level)

**Pipeline :**
1. Configuration & imports
2. Cache et utilitaires
3. Couche API YouTube
4. Phase B — Recherche des vidéos candidates
5. Phase C — Déduplication
6. Phase D — Enrichissement des métadonnées (batch)
7. Phase E — Filtrage qualité
8. Phase F — Collecte des commentaires
9. Résumé & vérification du dataset final

> Seed : 42 — Reproductible par tout tiers (NFR-02)

## 1 - Imports globaux

In [12]:
import os
import re
import sys
import json
import time
import logging
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
import subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import requests

print("Tous les imports effectués")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   requests: {requests.__version__}")

Tous les imports effectués
   pandas  : 2.3.3
   numpy   : 1.26.3
   requests: 2.28.1


## 2 — Configuration centrale

In [13]:
#  URLs des endpoints API
import os
from dotenv import load_dotenv
load_dotenv()  # Charge les variables depuis .env
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
if not YOUTUBE_API_KEY:
    raise ValueError("YOUTUBE_API_KEY manquante — vérifiez votre fichier .env")
YOUTUBE_API_BASE_URL     = "https://www.googleapis.com/youtube/v3"
ENDPOINT_SEARCH          = f"{YOUTUBE_API_BASE_URL}/search"
ENDPOINT_VIDEOS          = f"{YOUTUBE_API_BASE_URL}/videos"
ENDPOINT_COMMENT_THREADS = f"{YOUTUBE_API_BASE_URL}/commentThreads"

# Langues cibles
LANGUES_CIBLES = ["fr", "en"]

# ─── Requêtes de recherche — Contexte éducatif Cameroun ───────────────
REQUETES_PAR_THEMATIQUE = {
    "mathematiques": {
        "fr": [
            "mathématiques 3ème Cameroun",
            "cours maths troisième BEPC Cameroun",
            "exercices maths 3ème Cameroun corrigés",
            "algèbre 3ème Cameroun",
            "mathématiques Première Cameroun",
            "Probatoire maths Cameroun",
            "mathématiques Terminale Cameroun",
            "Baccalauréat maths Cameroun",
            "intégrales Terminale Cameroun",
            "nombres complexes Terminale Cameroun",
            "révision bac maths Cameroun",
        ],
        "en": [
            "mathematics Form 4 Cameroon",
            "GCE O-Level maths Cameroon",
            "algebra Form 4 Cameroon",
            "A-Level mathematics Lower 6 Cameroon",
            "calculus Lower 6th Cameroon",
            "A-Level mathematics Upper 6 Cameroon",
            "GCE A-Level revision maths Cameroon",
            "A-Level maths past questions Cameroon",
        ],
    },
    "svt_sciences": {
        "fr": [
            "SVT 3ème Cameroun",
            "BEPC SVT Cameroun révision",
            "biologie 3ème Cameroun cours",
            "SVT Première Cameroun",
            "Probatoire SVT Cameroun",
            "SVT Terminale Cameroun",
            "Baccalauréat SVT Cameroun",
            "révision bac SVT Cameroun",
        ],
        "en": [
            "biology Form 4 Cameroon",
            "chemistry O-Level Cameroon",
            "GCE O-Level science Cameroon",
            "biology Lower 6th Cameroon",
            "GCE A-Level biology Cameroon",
            "biology Upper 6th Cameroon",
            "GCE A-Level science revision Cameroon",
        ],
    },
    "physique_chimie": {
        "fr": [
            "physique chimie 3ème Cameroun",
            "BEPC physique Cameroun",
            "physique Première Cameroun",
            "Probatoire physique Cameroun",
            "physique Terminale Cameroun",
            "Baccalauréat physique Cameroun",
        ],
        "en": [
            "physics Form 5 Cameroon",
            "chemistry A-Level Cameroon",
            "GCE A-Level chemistry Cameroon",
        ],
    },
    "francais_anglais": {
        "fr": [
            "français 3ème Cameroun",
            "BEPC français Cameroun",
            "français Première Cameroun",
            "Probatoire français Cameroun",
            "Baccalauréat français Cameroun",
            "révision bac français Cameroun",
        ],
        "en": [
            "English language Form 4 Cameroon",
            "GCE O-Level English Cameroon",
            "English literature Lower 6th Cameroon",
            "GCE A-Level English Cameroon",
            "GCE A-Level literature Cameroon",
        ],
    },
    "histoire_geo_eco": {
        "fr": [
            "histoire géographie 3ème Cameroun",
            "BEPC histoire géo Cameroun",
            "histoire Première Cameroun",
            "Probatoire histoire géo Cameroun",
            "Baccalauréat histoire géo Cameroun",
            "économie Terminale Cameroun",
            "Baccalauréat économie Cameroun",
        ],
        "en": [
            "history Form 4 Cameroon",
            "GCE O-Level history Cameroon",
            "economics Lower 6th Cameroon",
            "GCE A-Level history Cameroon",
            "GCE A-Level economics Cameroon",
        ],
    },
}

# Paramètres de collecte
TARGET_COMMENTS            = 5000  # Objectif minimum de commentaires
MAX_RESULTATS_PAR_REQUETE  = 50    # Max par appel search.list (limite API : 50)
TAILLE_LOT_METADATA        = 50    # Taille des lots pour videos.list (max : 50)
MAX_COMMENTAIRES_PAR_VIDEO = 200   # Max commentaires collectés par vidéo

# Paramètres de filtrage (adaptés au contexte camerounais : chaînes moins populaires)
SEUIL_MIN_COMMENTAIRES = 5     # Vidéos avec moins de 5 commentaires éliminées
SEUIL_MIN_VUES         = 100   # Vidéos avec moins de 100 vues éliminées
DUREE_MIN_SECONDES     = 120   # Vidéos de moins de 2 min éliminées

# Paramètres de robustesse
DELAI_ENTRE_APPELS_SECONDES     = 0.5
MAX_TENTATIVES_API              = 3
DELAI_ENTRE_TENTATIVES_SECONDES = 2

# Répertoires et fichiers de sortie
from pathlib import Path
REPERTOIRE_DATA     = Path("data")
REPERTOIRE_DATA_RAW = REPERTOIRE_DATA / "raw"
REPERTOIRE_DATA.mkdir(parents=True, exist_ok=True)
REPERTOIRE_DATA_RAW.mkdir(parents=True, exist_ok=True)

FICHIER_RESULTATS_RECHERCHE_BRUTS      = REPERTOIRE_DATA_RAW / "search_results_raw.csv"
FICHIER_VIDEOS_CANDIDATES_DEDUPLIQUEES = REPERTOIRE_DATA_RAW / "videos_candidates_deduplicated.csv"
FICHIER_METADATA_VIDEOS                = REPERTOIRE_DATA_RAW / "videos_metadata.csv"
FICHIER_VIDEOS_SELECTIONNEES           = REPERTOIRE_DATA_RAW / "videos_selected.csv"
FICHIER_COMMENTAIRES_BRUTS             = REPERTOIRE_DATA_RAW / "comments_raw.csv"
FICHIER_CACHE_REQUETES                 = REPERTOIRE_DATA_RAW / "cache_requetes.json"
FICHIER_CACHE_VIDEOS_ENRICHIES         = REPERTOIRE_DATA_RAW / "cache_videos_enrichies.json"
FICHIER_CACHE_COMMENTAIRES             = REPERTOIRE_DATA_RAW / "cache_commentaires.json"

total_requetes = sum(len(q) for th in REQUETES_PAR_THEMATIQUE.values() for q in th.values())
print("\n Configuration chargée — Contexte : Éducation Cameroun")
print(f"   Thématiques    : {list(REQUETES_PAR_THEMATIQUE.keys())}")
print(f"   Langues cibles : {LANGUES_CIBLES}")
print(f"   Requêtes total : {total_requetes}")
print(f"   Objectif comms : {TARGET_COMMENTS}")
print(f"   Répertoire     : {REPERTOIRE_DATA_RAW}")



 Configuration chargée — Contexte : Éducation Cameroun
   Thématiques    : ['mathematiques', 'svt_sciences', 'physique_chimie', 'francais_anglais', 'histoire_geo_eco']
   Langues cibles : ['fr', 'en']
   Requêtes total : 66
   Objectif comms : 5000
   Répertoire     : data\raw


## 3 — Système de cache (persistance et reprise)

In [14]:
# SYSTÈME DE CACHE
# Permet de reprendre le pipeline sans répéter les appels déjà faits

def charger_cache(chemin_fichier: Path) -> Set[str]:
    """Charge un fichier de cache JSON. Retourne un set vide si inexistant."""
    if not chemin_fichier.exists():
        return set()
    try:
        with open(chemin_fichier, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except (json.JSONDecodeError, IOError):
        return set()  # Fichier corrompu (cache vide)

def sauvegarder_cache(chemin_fichier: Path, ensemble: Set[str]) -> None:
    """Sauvegarde un set de clés dans un fichier JSON."""
    with open(chemin_fichier, "w", encoding="utf-8") as f:
        json.dump(list(ensemble), f, ensure_ascii=False, indent=2)

def ajouter_au_cache(chemin_fichier: Path, cle: str) -> None:
    """Ajoute une clé au cache et sauvegarde immédiatement."""
    cache = charger_cache(chemin_fichier)
    cache.add(cle)
    sauvegarder_cache(chemin_fichier, cache)

def est_dans_cache(chemin_fichier: Path, cle: str) -> bool:
    """Vérifie si une clé est déjà dans le cache."""
    return cle in charger_cache(chemin_fichier)

#  Fonctions de cache spécialisées 

def requete_deja_executee(thematique: str, langue: str, requete: str) -> bool:
    """Vérifie si une requête de recherche a déjà été exécutée."""
    cle = f"{thematique}|{langue}|{requete}"
    return est_dans_cache(FICHIER_CACHE_REQUETES, cle)

def marquer_requete_executee(thematique: str, langue: str, requete: str) -> None:
    """Marque une requête de recherche comme exécutée."""
    cle = f"{thematique}|{langue}|{requete}"
    ajouter_au_cache(FICHIER_CACHE_REQUETES, cle)

def video_deja_enrichie(video_id: str) -> bool:
    """Vérifie si les métadonnées d'une vidéo ont déjà été collectées."""
    return est_dans_cache(FICHIER_CACHE_VIDEOS_ENRICHIES, video_id)

def marquer_videos_enrichies(video_ids: List[str]) -> None:
    """Marque un lot de vidéos comme enrichies."""
    cache = charger_cache(FICHIER_CACHE_VIDEOS_ENRICHIES)
    for vid in video_ids:
        cache.add(vid)
    sauvegarder_cache(FICHIER_CACHE_VIDEOS_ENRICHIES, cache)

def commentaires_deja_collectes(video_id: str) -> bool:
    """Vérifie si les commentaires d'une vidéo ont déjà été collectés."""
    return est_dans_cache(FICHIER_CACHE_COMMENTAIRES, video_id)

def marquer_commentaires_collectes(video_id: str) -> None:
    """Marque une vidéo comme ayant ses commentaires collectés."""
    ajouter_au_cache(FICHIER_CACHE_COMMENTAIRES, video_id)

print("Système de cache initialisé")

# Affichage de l'état du cache actuel
nb_requetes_en_cache   = len(charger_cache(FICHIER_CACHE_REQUETES))
nb_videos_en_cache     = len(charger_cache(FICHIER_CACHE_VIDEOS_ENRICHIES))
nb_comments_en_cache   = len(charger_cache(FICHIER_CACHE_COMMENTAIRES))
print(f"   Requêtes en cache     : {nb_requetes_en_cache}")
print(f"   Vidéos enrichies      : {nb_videos_en_cache}")
print(f"   Vidéos commentées     : {nb_comments_en_cache}")

Système de cache initialisé
   Requêtes en cache     : 66
   Vidéos enrichies      : 1717
   Vidéos commentées     : 288


## 4 — Fonctions utilitaires

In [15]:
# FONCTIONS UTILITAIRES

def configurer_logger(nom: str) -> logging.Logger:
    """Configure et retourne un logger formaté."""
    logger = logging.getLogger(nom)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter(
            "[%(asctime)s] %(levelname)s — %(message)s",
            datefmt="%H:%M:%S"
        ))
        logger.addHandler(handler)
    return logger

logger = configurer_logger("collecte")

def convertir_duree_iso_en_secondes(duree_iso: str) -> int:
    """
    Convertit une durée ISO 8601 en secondes.
    Exemples : 'PT4M33S' → 273 | 'PT1H2M5S' → 3725 | 'PT15S' → 15
    """
    if not duree_iso:
        return 0
    pattern = r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?"
    match = re.match(pattern, duree_iso)
    if not match:
        return 0
    heures   = int(match.group(1) or 0)
    minutes  = int(match.group(2) or 0)
    secondes = int(match.group(3) or 0)
    return heures * 3600 + minutes * 60 + secondes

def construire_url_youtube(video_id: str) -> str:
    """Retourne l'URL YouTube complète à partir d'un video_id."""
    return f"https://www.youtube.com/watch?v={video_id}"

# ─── Tests unitaires rapides ───────────────────────────────────────────
assert convertir_duree_iso_en_secondes("PT4M33S")  == 273
assert convertir_duree_iso_en_secondes("PT1H2M5S") == 3725
assert convertir_duree_iso_en_secondes("PT15S")    == 15
assert convertir_duree_iso_en_secondes("")         == 0

print(" Fonctions utilitaires opérationnelles (tests réussis)")

 Fonctions utilitaires opérationnelles (tests réussis)


## 5 — Couche d'accès à l'API YouTube

In [16]:

# COUCHE D'ACCÈS API — FONCTION PRINCIPALE D'APPEL HTTP

def executer_requete_api(endpoint: str, parametres: dict) -> Optional[dict]:
    """
    Exécute un appel GET vers l'API YouTube avec gestion des erreurs et retries.
    
    Codes d'erreur gérés :
      - 200 : succès
      - 400 : paramètres invalides (arrêt sans retry)
      - 403 : quota dépassé ou clé invalide (arrêt sans retry)
      - 429 / 503 : rate limiting ou indispo temporaire (retry avec délai)
    """
    parametres_complets = {"key": YOUTUBE_API_KEY, **parametres}

    for tentative in range(1, MAX_TENTATIVES_API + 1):
        time.sleep(DELAI_ENTRE_APPELS_SECONDES)

        try:
            reponse = requests.get(endpoint, params=parametres_complets, timeout=15)

            if reponse.status_code == 200:
                return reponse.json()

            elif reponse.status_code == 403:
                raison = reponse.json().get("error", {}).get("message", "?")
                logger.error(f" Accès refusé (403) : {raison}")
                return None  
            elif reponse.status_code == 400:
                raison = reponse.json().get("error", {}).get("message", "?")
                logger.error(f" Requête invalide (400) : {raison}")
                return None  
            elif reponse.status_code in (429, 503):
                logger.warning(f" Code {reponse.status_code} — retry {tentative}/{MAX_TENTATIVES_API}")
                time.sleep(DELAI_ENTRE_TENTATIVES_SECONDES)

            else:
                logger.warning(f" HTTP {reponse.status_code} — retry {tentative}/{MAX_TENTATIVES_API}")
                time.sleep(DELAI_ENTRE_TENTATIVES_SECONDES)

        except requests.exceptions.Timeout:
            logger.warning(f"  Timeout — retry {tentative}/{MAX_TENTATIVES_API}")
            time.sleep(DELAI_ENTRE_TENTATIVES_SECONDES)

        except requests.exceptions.ConnectionError as e:
            logger.warning(f"  Erreur réseau : {e} — retry {tentative}/{MAX_TENTATIVES_API}")
            time.sleep(DELAI_ENTRE_TENTATIVES_SECONDES)

    logger.error(f" Échec après {MAX_TENTATIVES_API} tentatives : {endpoint}")
    return None


#  Fonctions API spécialisées 

def api_rechercher_videos(requete: str, langue: str, max_resultats: int = 50) -> Optional[dict]:
    """Appelle search.list. Coût : 100 unités de quota. À minimiser """
    return executer_requete_api(ENDPOINT_SEARCH, {
        "q":                 requete,
        "part":              "snippet",
        "type":              "video",
        "maxResults":        min(max_resultats, 50),
        "relevanceLanguage": langue,
        "safeSearch":        "moderate",
    })

def api_metadata_videos_batch(lot_video_ids: List[str]) -> Optional[dict]:
    """Appelle videos.list pour un lot de 50 vidéos max. Coût : 1 unité."""
    return executer_requete_api(ENDPOINT_VIDEOS, {
        "id":   ",".join(lot_video_ids[:50]),
        "part": "snippet,statistics,contentDetails",
    })

def api_commentaires_page(
    video_id: str,
    max_resultats: int = 100,
    jeton_page: Optional[str] = None
) -> Optional[dict]:
    """Appelle commentThreads.list. Coût : 1 unité."""
    params = {
        "videoId":    video_id,
        "part":       "snippet",
        "maxResults": min(max_resultats, 100),
        "order":      "relevance",  # Commentaires les plus utiles en premier
    }
    if jeton_page:
        params["pageToken"] = jeton_page
    return executer_requete_api(ENDPOINT_COMMENT_THREADS, params)

print(" Couche d'accès API configurée")
print(f"   Endpoint search  : {ENDPOINT_SEARCH}")
print(f"   Endpoint videos  : {ENDPOINT_VIDEOS}")
print(f"   Endpoint comments: {ENDPOINT_COMMENT_THREADS}")

 Couche d'accès API configurée
   Endpoint search  : https://www.googleapis.com/youtube/v3/search
   Endpoint videos  : https://www.googleapis.com/youtube/v3/videos
   Endpoint comments: https://www.googleapis.com/youtube/v3/commentThreads


In [17]:
# import json

# with open(FICHIER_CACHE_REQUETES, "w", encoding="utf-8") as f:
#     json.dump({}, f, indent=4, ensure_ascii=False)

# print(" Cache des requêtes réinitialisé")

##  6 — Phase B : Recherche des vidéos candidates

In [18]:

# PHASE B — RECHERCHE DES VIDÉOS CANDIDATES (search.list)
# Coût quota : 100 unités par requête et  à minimiser absolument


COLONNES_RECHERCHE = ["video_id","titre","chaine","publie_le","requete_utilisee","langue","thematique"]

def executer_phase_recherche() -> pd.DataFrame:
    """Exécute toutes les requêtes de recherche par thématique et langue."""

    logger.info(" Phase B — Recherche des vidéos candidates")

    # Construire la liste de toutes les requêtes
    toutes_requetes = [
        (thematique, langue, requete)
        for thematique, langues in REQUETES_PAR_THEMATIQUE.items()
        for langue in LANGUES_CIBLES
        if langue in langues
        for requete in langues[langue]
    ]

    logger.info(f" {len(toutes_requetes)} requêtes planifiées")
    nb_nouvelles = 0

    for thematique, langue, requete in tqdm(toutes_requetes, desc="Recherche vidéos"):

        # VÉRIFICATION CACHE 
        if requete_deja_executee(thematique, langue, requete):
            print(" Requête ignorée (déjà exécutée)")
            continue  

        # APPEL API 
        reponse = api_rechercher_videos(requete, langue, MAX_RESULTATS_PAR_REQUETE)
        marquer_requete_executee(thematique, langue, requete)  # Toujours marquer

        if not reponse:
            continue

        #  EXTRACTION DES RÉSULTATS
        lignes = []
        for item in reponse.get("items", []):
            if item.get("id", {}).get("kind") != "youtube#video":
                continue
            try:
                lignes.append({
                    "video_id":         item["id"]["videoId"],
                    "titre":            item["snippet"].get("title", ""),
                    "chaine":           item["snippet"].get("channelTitle", ""),
                    "publie_le":        item["snippet"].get("publishedAt", ""),
                    "requete_utilisee": requete,
                    "langue":           langue,
                    "thematique":       thematique,
                })
            except (KeyError, TypeError):
                continue

        # SAUVEGARDE IMMÉDIATE
        if lignes:
            df_batch = pd.DataFrame(lignes, columns=COLONNES_RECHERCHE)
            mode = "a" if FICHIER_RESULTATS_RECHERCHE_BRUTS.exists() else "w"
            df_batch.to_csv(FICHIER_RESULTATS_RECHERCHE_BRUTS, mode=mode,
                           header=(mode=="w"), index=False, encoding="utf-8")
            nb_nouvelles += len(lignes)

    # Charger et retourner le fichier complet
    if FICHIER_RESULTATS_RECHERCHE_BRUTS.exists():
        df = pd.read_csv(FICHIER_RESULTATS_RECHERCHE_BRUTS, encoding="utf-8")
        logger.info(f" Phase B terminée : {len(df)} vidéos candidates ({nb_nouvelles} nouvelles)")
        return df
    return pd.DataFrame(columns=COLONNES_RECHERCHE)


# EXÉCUTION 
df_resultats_bruts = executer_phase_recherche()
print(f"\n Résultats bruts : {len(df_resultats_bruts)} lignes")
df_resultats_bruts.head()

[13:03:27] INFO —  Phase B — Recherche des vidéos candidates
[13:03:27] INFO —  66 requêtes planifiées


Recherche vidéos:   0%|          | 0/66 [00:00<?, ?it/s]

 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête ignorée (déjà exécutée)
 Requête i

Recherche vidéos: 100%|██████████| 66/66 [00:00<00:00, 2932.92it/s]

[13:03:27] INFO —  Phase B terminée : 3082 vidéos candidates (0 nouvelles)

 Résultats bruts : 3082 lignes


,video_id,titre,chaine,publie_le,requete_utilisee,langue,thematique
0,2dwMa7aeb0M,Développement simple - Maths 3e - Les Bons Profs,Les Bons Profs,2014-06-19T16:57:28Z,mathématiques 3ème Cameroun,fr,mathematiques
1,JDF2BVeFoZ8,Démontrons que 1+1=3,Éducation Plus,2024-09-08T08:50:26Z,mathématiques 3ème Cameroun,fr,mathematiques
2,XqkEQ_jmiDU,Cours - Troisième - Mathématiques : Relation t...,Ecoles Au Senegal,2020-08-06T12:53:48Z,mathématiques 3ème Cameroun,fr,mathematiques
3,IMryb56BpsQ,IP-SC MATHEMATIQUES 3eme Révisions Phase 2 Cal...,ENSEIGNEMENT A DISTANCE MINESEC,2021-03-27T10:20:35Z,mathématiques 3ème Cameroun,fr,mathematiques
4,riBTJya57-Q,Notion de fonction - Maths 3e - Les Bons Profs,Les Bons Profs,2013-12-09T15:47:38Z,mathématiques 3ème Cameroun,fr,mathematiques


##  7 — Phase C : Déduplication des vidéos

In [19]:
# PHASE C — DÉDUPLICATION SUR video_id


logger.info(" Phase C — Déduplication des vidéos")

nb_avant = len(df_resultats_bruts)
df_deduplique = df_resultats_bruts.drop_duplicates(subset=["video_id"], keep="first").reset_index(drop=True)
nb_apres  = len(df_deduplique)

logger.info(f" {nb_avant}  {nb_apres} vidéos ({nb_avant - nb_apres} doublons supprimés)")

# Sauvegarde
df_deduplique.to_csv(FICHIER_VIDEOS_CANDIDATES_DEDUPLIQUEES, index=False, encoding="utf-8")
logger.info(f" Sauvegardé  {FICHIER_VIDEOS_CANDIDATES_DEDUPLIQUEES.name}")

# Affichage répartition
print("\n Répartition après déduplication :")
print(df_deduplique.groupby(["thematique","langue"]).size().reset_index(name="nb_videos").to_string(index=False))

[13:03:27] INFO —  Phase C — Déduplication des vidéos
[13:03:27] INFO —  3082  1717 vidéos (1365 doublons supprimés)
[13:03:27] INFO —  Sauvegardé  videos_candidates_deduplicated.csv

 Répartition après déduplication :
      thematique langue  nb_videos
francais_anglais     en        128
francais_anglais     fr         92
histoire_geo_eco     en        168
histoire_geo_eco     fr        149
   mathematiques     en        213
   mathematiques     fr        332
 physique_chimie     en         58
 physique_chimie     fr        176
    svt_sciences     en        204
    svt_sciences     fr        197


## 8 — Phase D : Enrichissement batch des métadonnées

In [20]:

# PHASE D — ENRICHISSEMENT BATCH (videos.list)
# 1 appel pour 50 vidéos = économie massive de quota


COLONNES_METADATA = [
    "video_id","video_url","titre","chaine","publie_le",
    "categorie_id","description","nb_vues","nb_likes",
    "nb_commentaires","duree_iso","duree_secondes","langue","thematique"
]

def executer_phase_metadata(df_candidates: pd.DataFrame) -> pd.DataFrame:
    """Enrichit toutes les vidéos candidates en batch (50 vidéos par appel)."""

    logger.info(" Phase D — Enrichissement batch des métadonnées")

    # Mapping video_id → (langue, thématique)
    mapping = {row["video_id"]: (row["langue"], row["thematique"]) for _, row in df_candidates.iterrows()}

    # Filtrer les vidéos non encore enrichies
    videos_a_traiter = [vid for vid in df_candidates["video_id"].tolist() if not video_deja_enrichie(vid)]
    logger.info(f" {len(videos_a_traiter)}/{len(df_candidates)} vidéos à enrichir")

    # Division en lots de 50
    lots = [videos_a_traiter[i:i+TAILLE_LOT_METADATA] for i in range(0, len(videos_a_traiter), TAILLE_LOT_METADATA)]
    logger.info(f" {len(lots)} lot(s) de {TAILLE_LOT_METADATA} vidéos")

    for i, lot in enumerate(tqdm(lots, desc="Enrichissement batch"), 1):
        reponse = api_metadata_videos_batch(lot)
        if not reponse:
            continue

        lignes = []
        for item in reponse.get("items", []):
            try:
                vid_id     = item["id"]
                snippet    = item.get("snippet", {})
                stats      = item.get("statistics", {})
                details    = item.get("contentDetails", {})
                duree_iso  = details.get("duration", "")
                langue, thematique = mapping.get(vid_id, ("inconnu", "inconnu"))

                lignes.append({
                    "video_id":        vid_id,
                    "video_url":       construire_url_youtube(vid_id),
                    "titre":           snippet.get("title", ""),
                    "chaine":          snippet.get("channelTitle", ""),
                    "publie_le":       snippet.get("publishedAt", ""),
                    "categorie_id":    snippet.get("categoryId", ""),
                    "description":     snippet.get("description", "")[:300],
                    "nb_vues":         int(stats.get("viewCount", 0)),
                    "nb_likes":        int(stats.get("likeCount", 0)),
                    "nb_commentaires": int(stats.get("commentCount", 0)),
                    "duree_iso":       duree_iso,
                    "duree_secondes":  convertir_duree_iso_en_secondes(duree_iso),
                    "langue":          langue,
                    "thematique":      thematique,
                })
            except (KeyError, TypeError, ValueError):
                continue

        if lignes:
            df_lot = pd.DataFrame(lignes, columns=COLONNES_METADATA)
            mode = "a" if FICHIER_METADATA_VIDEOS.exists() else "w"
            df_lot.to_csv(FICHIER_METADATA_VIDEOS, mode=mode,
                         header=(mode=="w"), index=False, encoding="utf-8")

        marquer_videos_enrichies(lot)
        logger.info(f"  Lot {i}/{len(lots)} → {len(lignes)} métadonnées sauvegardées")

    if FICHIER_METADATA_VIDEOS.exists():
        df = pd.read_csv(FICHIER_METADATA_VIDEOS, encoding="utf-8")
        logger.info(f" Phase D terminée : {len(df)} vidéos enrichies")
        return df
    return pd.DataFrame(columns=COLONNES_METADATA)


# EXÉCUTION 
df_metadata = executer_phase_metadata(df_deduplique)
print(f"\n Métadonnées collectées : {len(df_metadata)} vidéos")
df_metadata[["titre","nb_vues","nb_commentaires","duree_secondes","langue","thematique"]].head(10)

[13:03:27] INFO —  Phase D — Enrichissement batch des métadonnées


[13:03:28] INFO —  0/1717 vidéos à enrichir
[13:03:28] INFO —  0 lot(s) de 50 vidéos


Enrichissement batch: 0it [00:00, ?it/s]

[13:03:28] INFO —  Phase D terminée : 1717 vidéos enrichies

 Métadonnées collectées : 1717 vidéos


,titre,nb_vues,nb_commentaires,duree_secondes,langue,thematique
0,Développement simple - Maths 3e - Les Bons Profs,233155,132,191,fr,mathematiques
1,Démontrons que 1+1=3,1112226,2453,327,fr,mathematiques
2,Cours - Troisième - Mathématiques : Relation t...,72337,94,779,fr,mathematiques
3,IP-SC MATHEMATIQUES 3eme Révisions Phase 2 Cal...,750,1,5697,fr,mathematiques
4,Notion de fonction - Maths 3e - Les Bons Profs,594156,576,252,fr,mathematiques
5,Factorisation,569888,1480,1102,fr,mathematiques
6,Théorème de Pythagore - Maths 3e - Les Bons Profs,408717,401,185,fr,mathematiques
7,Révision - composition du premier semestre - T...,88678,64,1650,fr,mathematiques
8,Théorème de Thalès,127268,274,894,fr,mathematiques
9,Cours complet - troisième -sur les racines car...,162879,364,4274,fr,mathematiques


##  9 — Phase E : Filtrage des vidéos

In [21]:
# PHASE E — FILTRAGE DES VIDÉOS PEU EXPLOITABLES
# Filtrer AVANT de collecter les commentaires = économie de quota


logger.info(" Phase E — Filtrage des vidéos")

df_filtre = df_metadata.copy()

# Conversion numérique sécurisée
for col in ["nb_commentaires", "nb_vues", "duree_secondes"]:
    df_filtre[col] = pd.to_numeric(df_filtre[col], errors="coerce").fillna(0)

nb_initial = len(df_filtre)

# Règle F1 : commentaires minimum
df_filtre = df_filtre[df_filtre["nb_commentaires"] >= SEUIL_MIN_COMMENTAIRES]
logger.info(f"F1 (≥{SEUIL_MIN_COMMENTAIRES} commentaires) : {nb_initial} → {len(df_filtre)}")

#  Règle F2 : vues minimum 
n2 = len(df_filtre)
df_filtre = df_filtre[df_filtre["nb_vues"] >= SEUIL_MIN_VUES]
logger.info(f"F2 (≥{SEUIL_MIN_VUES} vues)      : {n2} → {len(df_filtre)}")

#  Règle F3 : durée minimale 
n3 = len(df_filtre)
df_filtre = df_filtre[df_filtre["duree_secondes"] >= DUREE_MIN_SECONDES]
logger.info(f"F3 (≥{DUREE_MIN_SECONDES}s durée)      : {n3} → {len(df_filtre)}")

df_filtre = df_filtre.reset_index(drop=True)
df_filtre.to_csv(FICHIER_VIDEOS_SELECTIONNEES, index=False, encoding="utf-8")

logger.info(f" Phase E terminée : {nb_initial} → {len(df_filtre)} vidéos retenues")
logger.info(f" Sauvegardé → {FICHIER_VIDEOS_SELECTIONNEES.name}")

print("\n Répartition des vidéos sélectionnées :")
print(df_filtre.groupby(["thematique","langue"]).size().reset_index(name="nb_videos").to_string(index=False))

[13:03:28] INFO —  Phase E — Filtrage des vidéos
[13:03:28] INFO — F1 (≥5 commentaires) : 1717 → 963
[13:03:28] INFO — F2 (≥100 vues)      : 963 → 963
[13:03:28] INFO — F3 (≥120s durée)      : 963 → 591
[13:03:28] INFO —  Phase E terminée : 1717 → 591 vidéos retenues
[13:03:28] INFO —  Sauvegardé → videos_selected.csv

 Répartition des vidéos sélectionnées :
      thematique langue  nb_videos
francais_anglais     en         23
francais_anglais     fr         34
histoire_geo_eco     en         35
histoire_geo_eco     fr         46
   mathematiques     en         66
   mathematiques     fr        149
 physique_chimie     en         13
 physique_chimie     fr         83
    svt_sciences     en         60
    svt_sciences     fr         82


##  10 — Phase F : Collecte des commentaires

In [22]:
# PHASE F — COLLECTE DES COMMENTAIRES (commentThreads.list)
# Uniquement pour les vidéos retenues après filtrage


COLONNES_COMMENTAIRES = [
    "commentaire_id","video_id","texte_commentaire",
    "publie_le","nb_likes_commentaire","nb_reponses"
]

def collecter_commentaires_une_video(video_id: str) -> List[Dict]:
    """Collecte les commentaires top-level d'une vidéo avec pagination contrôlée."""
    commentaires   = []
    jeton_page     = None
    continuer      = True

    while continuer:
        nb_restants = MAX_COMMENTAIRES_PAR_VIDEO - len(commentaires)
        if nb_restants <= 0:
            break

        reponse = api_commentaires_page(video_id, min(nb_restants, 100), jeton_page)
        if not reponse:
            break  # Commentaires désactivés ou erreur → passer

        for item in reponse.get("items", []):
            try:
                snippet_top = item["snippet"]["topLevelComment"]["snippet"]
                texte = snippet_top.get("textDisplay", "").strip()
                if len(texte) < 3:
                    continue  # Ignorer les commentaires trop courts

                commentaires.append({
                    "commentaire_id":       item["id"],
                    "video_id":             video_id,
                    "texte_commentaire":    texte,
                    "publie_le":            snippet_top.get("publishedAt", ""),
                    "nb_likes_commentaire": int(snippet_top.get("likeCount", 0)),
                    "nb_reponses":          int(item["snippet"].get("totalReplyCount", 0)),
                })
            except (KeyError, TypeError):
                continue

            if len(commentaires) >= MAX_COMMENTAIRES_PAR_VIDEO:
                continuer = False
                break

        jeton_page = reponse.get("nextPageToken")
        if not jeton_page:
            continuer = False

    return commentaires


def executer_phase_commentaires(df_videos: pd.DataFrame) -> pd.DataFrame:
    """Collecte les commentaires pour toutes les vidéos sélectionnées."""

    logger.info("Phase F — Collecte des commentaires")

    video_ids = df_videos["video_id"].tolist()
    a_traiter = [v for v in video_ids if not commentaires_deja_collectes(v)]
    logger.info(f" {len(a_traiter)}/{len(video_ids)} vidéos à traiter")

    nb_total = 0

    for video_id in tqdm(a_traiter, desc="Collecte commentaires"):
        commentaires = collecter_commentaires_une_video(video_id)
        marquer_commentaires_collectes(video_id)  

        if commentaires:
            df_comm = pd.DataFrame(commentaires, columns=COLONNES_COMMENTAIRES)
            mode = "a" if FICHIER_COMMENTAIRES_BRUTS.exists() else "w"
            df_comm.to_csv(FICHIER_COMMENTAIRES_BRUTS, mode=mode,
                          header=(mode=="w"), index=False, encoding="utf-8")
            nb_total += len(commentaires)

    if FICHIER_COMMENTAIRES_BRUTS.exists():
        df = pd.read_csv(FICHIER_COMMENTAIRES_BRUTS, encoding="utf-8")
        logger.info(f" Phase F terminée : {len(df)} commentaires collectés")
        return df
    return pd.DataFrame(columns=COLONNES_COMMENTAIRES)


#  EXÉCUTION 
df_commentaires = executer_phase_commentaires(df_filtre)
print(f"\n Commentaires collectés : {len(df_commentaires)}")
df_commentaires.head(10)

[13:03:28] INFO — Phase F — Collecte des commentaires
[13:03:28] INFO —  303/591 vidéos à traiter


Collecte commentaires: 100%|██████████| 303/303 [10:56<00:00,  2.17s/it]

[13:14:25] INFO —  Phase F terminée : 42860 commentaires collectés

 Commentaires collectés : 42860


,commentaire_id,video_id,texte_commentaire,publie_le,nb_likes_commentaire,nb_reponses
0,UgxoD16zA6d6P-Gu_jp4AaABAg,2dwMa7aeb0M,Moi je suis nulle en maths mais grâce à vos vi...,2025-10-12T17:56:42Z,9,0
1,UghvGEzou_Ana3gCoAEC,2dwMa7aeb0M,grâce a vous je suis devenu plus fort en maths...,2016-04-21T16:36:18Z,22,0
2,UgxAugZhHsUnjyRqWaZ4AaABAg,2dwMa7aeb0M,je suis en 3e et heureuse d&#39;avoir découver...,2018-03-29T19:26:40Z,38,6
3,Ugg0yvVZvm9KangCoAEC,2dwMa7aeb0M,J&#39;ai toujours eu beaucoup de mal en mathém...,2016-09-23T19:09:38Z,55,5
4,Ugjhos2JSgNa1XgCoAEC,2dwMa7aeb0M,"Merci bcp, les explications sont claires, ça a...",2016-05-29T17:50:28Z,27,0
5,UgwkLFzWC9FpdeYOds94AaABAg,2dwMa7aeb0M,Vous êtes géniaux ! Grâce à votre chaîne youtu...,2018-11-28T15:23:29Z,7,0
6,Ugyx-9y3Go5gj5iD3yZ4AaABAg,2dwMa7aeb0M,Meilleur prof de l’humanité,2020-02-12T22:01:35Z,5,0
7,UgxaphHIghRn7l4hTnN4AaABAg,2dwMa7aeb0M,Merci beaucoup monsieur vous m&#39;avez beauco...,2020-12-15T19:16:39Z,6,0
8,Ugy0Nsd7ah8ZXEuJgPR4AaABAg,2dwMa7aeb0M,Super🤗je suis une maman qui se remet a niveau ...,2018-12-02T23:20:41Z,6,0
9,UggWQqNUgdjGHngCoAEC,2dwMa7aeb0M,"Sympa les exemples nombreux, ils aident bien a...",2015-01-05T10:53:23Z,8,0


##   11 — Résumé et vérification du dataset final

In [23]:

# RÉSUMÉ FINAL DE LA COLLECTE


print("=" * 55)
print("   RÉSUMÉ DE LA COLLECTE")
print("=" * 55)
print(f"  Vidéos candidates (brutes)     : {len(df_resultats_bruts)}")
print(f"  Vidéos après déduplication     : {len(df_deduplique)}")
print(f"  Vidéos enrichies (métadonnées) : {len(df_metadata)}")
print(f"  Vidéos sélectionnées (filtrées): {len(df_filtre)}")
print(f"  Commentaires collectés         : {len(df_commentaires)}")
print("=" * 55)

#  Vérification des fichiers produits 
print("\n Fichiers produits :")
for fichier in [
    FICHIER_RESULTATS_RECHERCHE_BRUTS,
    FICHIER_VIDEOS_CANDIDATES_DEDUPLIQUEES,
    FICHIER_METADATA_VIDEOS,
    FICHIER_VIDEOS_SELECTIONNEES,
    FICHIER_COMMENTAIRES_BRUTS,
]:
    taille = fichier.stat().st_size / 1024 if fichier.exists() else 0
    statut = "OKK" if fichier.exists() else "NOO"
    print(f"  {statut} {fichier.name:<50} ({taille:.1f} Ko)")

# Aperçu du dataset commentaires final 
print("\n Aperçu du dataset de commentaires :")
print(f"   Colonnes : {list(df_commentaires.columns)}")
print(f"   Taille   : {df_commentaires.shape}")
print(f"   Mémoire  : {df_commentaires.memory_usage().sum() / 1024:.1f} Ko")

# Distribution des commentaires par vidéo 
if not df_commentaires.empty:
    stats_par_video = df_commentaires.groupby("video_id")["texte_commentaire"].count()
    print(f"\n Distribution commentaires/vidéo :")
    print(f"   Min    : {stats_par_video.min()}")
    print(f"   Médiane: {stats_par_video.median():.1f}")
    print(f"   Max    : {stats_par_video.max()}")
    print(f"   Moyenne: {stats_par_video.mean():.1f}")

   RÉSUMÉ DE LA COLLECTE
  Vidéos candidates (brutes)     : 3082
  Vidéos après déduplication     : 1717
  Vidéos enrichies (métadonnées) : 1717
  Vidéos sélectionnées (filtrées): 591
  Commentaires collectés         : 42860

 Fichiers produits :
  OKK search_results_raw.csv                             (504.0 Ko)
  OKK videos_candidates_deduplicated.csv                 (281.9 Ko)
  OKK videos_metadata.csv                                (633.7 Ko)
  OKK videos_selected.csv                                (251.7 Ko)
  OKK comments_raw.csv                                   (6119.9 Ko)

 Aperçu du dataset de commentaires :
   Colonnes : ['commentaire_id', 'video_id', 'texte_commentaire', 'publie_le', 'nb_likes_commentaire', 'nb_reponses']
   Taille   : (42860, 6)
   Mémoire  : 2009.2 Ko

 Distribution commentaires/vidéo :
   Min    : 2
   Médiane: 40.0
   Max    : 200
   Moyenne: 72.5



##  Collecte terminée

Next step : 02 — Preprocessing 

| Fichier | Contenu |
|---|---|
| `videos_metadata.csv` | Métadonnées complètes des vidéos sélectionnées |
| `comments_raw.csv` | Commentaires bruts à analyser |
| **Réalisée par**| ***Kevin***|
|**Email institutionnelle**| stive.watat@facsciences-uy1.cm|
|**Email personnel**|   stivekevinwatatyondep@gmail.com| 

**Prochaine étape** : `02_preprocessing_and_noise.ipynb`( Nettoyage et filtrage du bruit)


